In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_openai import OpenAI, ChatOpenAI
from langchain.schema import HumanMessage
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
import os
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

In [13]:
# Create LLM class
gemini = ChatGoogleGenerativeAI(
    model= "gemini-2.5-pro",
    temperature=1.0,
    max_retries=2,
    google_api_key=api_key,
)

In [14]:
llm1 = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
)

mistral = ChatHuggingFace(llm=llm1)

In [15]:
class BlogState(TypedDict):
    topic: str
    blogoutline: str
    final: str

In [16]:
def get_outline(state: BlogState) -> BlogState:
    print("Generating blog outline...")
    response = mistral.invoke([
        HumanMessage(content=f"Create a detailed blog outline on the topic: {state['topic']}")
    ])
    state['blogoutline'] = response.content
    print(state['blogoutline'])
    return state

In [17]:
def get_blog(state: BlogState) -> BlogState:
    print("Generating final blog post...")
    blogoutline = state["blogoutline"]
    response = gemini.invoke([
        HumanMessage(content=f"Write a detailed blog post based on the following outline: {blogoutline}")
    ])
    state['final'] = response.content
    return state

In [18]:
#define graph
graph = StateGraph(BlogState)

In [19]:
# add nodes
graph.add_node("outlineblog", get_outline)
graph.add_node("finalblog", get_blog)

In [20]:
#add edges
graph.add_edge(START, "outlineblog")
graph.add_edge("outlineblog", "finalblog")
graph.add_edge("finalblog", END)

In [21]:
#compile graph
workflow = graph.compile()

In [22]:
#execute workflow
initial_state = BlogState(topic="The Future of Artificial Intelligence in Everyday Life", blogoutline="", final="")
final_state = workflow.invoke(initial_state)

Generating blog outline...
 Title: The Future of Artificial Intelligence in Everyday Life: A Comprehensive Outlook

I. Introduction
  A. Brief explanation of Artificial Intelligence (AI)
  B. Importance of AI in today's world
  C. Preview of the future of AI in everyday life

II. Current Applications of AI in Everyday Life
  A. Home automation and IoT devices
      1. Smart homes and virtual assistants (Amazon Echo, Google Home)
      2. Security systems and surveillance
      3. Energy management and climate control
  B. Healthcare and telemedicine
      1. AI-assisted diagnosis and treatment plans
      2. Remote patient monitoring
      3. Mental health support through chatbots
  C. Education and e-learning
      1. Personalized learning and adaptive technology
      2. Intelligent tutoring systems
      3. Automated assessments and grading
  D. Transportation and logistics
      1. Autonomous vehicles
      2. Route optimization and traffic flow management
      3. Predictive maint

In [23]:
final_state['topic']

'The Future of Artificial Intelligence in Everyday Life'

In [24]:
final_state['blogoutline']

" Title: The Future of Artificial Intelligence in Everyday Life: A Comprehensive Outlook\n\nI. Introduction\n  A. Brief explanation of Artificial Intelligence (AI)\n  B. Importance of AI in today's world\n  C. Preview of the future of AI in everyday life\n\nII. Current Applications of AI in Everyday Life\n  A. Home automation and IoT devices\n      1. Smart homes and virtual assistants (Amazon Echo, Google Home)\n      2. Security systems and surveillance\n      3. Energy management and climate control\n  B. Healthcare and telemedicine\n      1. AI-assisted diagnosis and treatment plans\n      2. Remote patient monitoring\n      3. Mental health support through chatbots\n  C. Education and e-learning\n      1. Personalized learning and adaptive technology\n      2. Intelligent tutoring systems\n      3. Automated assessments and grading\n  D. Transportation and logistics\n      1. Autonomous vehicles\n      2. Route optimization and traffic flow management\n      3. Predictive maintena

In [25]:
final_state['final']

'Of course! Here is a detailed blog post based on the comprehensive outline you provided.\n\n***\n\n# The Future of Artificial Intelligence in Everyday Life: A Comprehensive Outlook\n\nOnce the realm of science fiction, Artificial Intelligence (AI) is no longer a far-off concept—it’s a present-day reality woven into the fabric of our daily routines. From the moment you ask your smart speaker for the weather to the personalized movie recommendation you receive at night, AI is working silently in the background. But what is this technology, and where is it taking us?\n\nThis post will explore the comprehensive landscape of AI, starting with how it\'s already shaping our world and then looking ahead to the groundbreaking advancements that will redefine our future.\n\n### What is Artificial Intelligence?\n\nAt its core, Artificial Intelligence is a branch of computer science focused on building smart machines capable of performing tasks that typically require human intelligence. This inclu